## Model Selection: Why Logistic Regression, Random Forest, and XGBoost

We train three models spanning a deliberate range of complexity and interpretability, rather than committing to a single algorithm upfront:

- **Logistic Regression** is our interpretable baseline. Its coefficients are directly readable as the (linear) direction and strength of each feature's effect on risk, which matters in a clinical context where a model that can't be explained to a clinician is hard to trust — and with only ~300 rows and mostly monotonic clinical risk factors, a linear model is a genuinely competitive candidate, not just a strawman.
- **Random Forest** captures non-linear relationships and feature interactions (e.g. the `hr_reserve` feature engineered in [01_preprocessing.ipynb](01_preprocessing.ipynb) only matters differently across `age` ranges) that logistic regression can't, while still exposing feature importances and being relatively robust to the outliers we saw in `chol`/`trestbps` during EDA.
- **XGBoost** gives us a stronger, regularized gradient-boosted alternative to compare Random Forest against — if boosting doesn't outperform bagging on this dataset, that's itself a useful finding on a dataset this small (boosting typically needs more data to earn its extra complexity).

Comparing all three under identical cross-validation (below) lets the data decide which trade-off between interpretability and predictive power is worth making, instead of assuming it.

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib

# Load the cleaned dataset
df = pd.read_csv('../data/processed/heart_cleaned.csv')
X_train = df.drop('target', axis=1)
y_train = df['target']

# Initialize the models
log_model = LogisticRegression(random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
xgb_model = XGBClassifier(random_state=42)

# Train the models
log_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

# Export the trained models
joblib.dump(log_model, '../models/logistic_model.pkl')
joblib.dump(rf_model, '../models/rf_model.pkl')
joblib.dump(xgb_model, '../models/xgb_model.pkl')

['../models/xgb_model.pkl']

## Model Comparison (Stratified K-Fold Cross-Validation)

Compare Logistic Regression, Random Forest, and XGBoost using 5-fold stratified cross-validation on accuracy, precision, recall, F1, and ROC-AUC.

In [2]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
}

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc',
}

results = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    results.append({
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1': scores['test_f1'].mean(),
        'ROC-AUC': scores['test_roc_auc'].mean(),
    })

comparison_df = pd.DataFrame(results).set_index('Model').round(4)
comparison_df

,Accuracy,Precision,Recall,F1,ROC-AUC
Model,,,,,
Logistic Regression,0.8054,0.8294,0.7364,0.7684,0.8962
Random Forest,0.8223,0.8378,0.7818,0.8007,0.9088
XGBoost,0.8058,0.8054,0.7636,0.7813,0.8943


## Hyperparameter Tuning: Random Forest (GridSearchCV, optimizing Recall)

Random Forest is the model we tune further, since it had the strongest balance of accuracy/precision/recall/ROC-AUC in the comparison above without XGBoost's added complexity.

**Why recall, not accuracy or F1, as the scoring metric:** in a cardiovascular screening tool, a false negative (telling an at-risk patient they're low-risk) is far more costly than a false positive (flagging a healthy patient for a closer look that turns out fine) — a missed diagnosis can mean a missed opportunity to intervene before a cardiac event, while a false alarm just costs a follow-up test. Optimizing for recall directly minimizes missed-diagnosis cases, even at some cost to precision.

We tune with `GridSearchCV` using `StratifiedKFold(n_splits=5)`, scoring on recall. The best estimator overwrites `models/rf_model.pkl`.

In [3]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

tuning_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring='recall',
    cv=tuning_cv,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best recall score: {grid_search.best_score_:.4f}")

rf_model = grid_search.best_estimator_
joblib.dump(rf_model, '../models/rf_model.pkl')

Best parameters: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Best recall score: 0.8091


['../models/rf_model.pkl']